# Gold de faturas — V-Energy

Consolida faturamento por **competência**, preserva o detalhe da conta e compara receita entre períodos. Lê apenas Silver Delta. Regras e limites: [dicionário](../../docs/gold/faturas.md).

O padrão `modo=validar` não grava dados. No Job use `modo=publicar`, informe a data de referência do retrato de status e um diretório em Volume para exportação. Execute o notebook inteiro; as fontes são fixadas em versões Delta.

In [ ]:
%run ./gold_faturas

In [ ]:
import re
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

parametros = {
    'tabela_faturas': 'v-energy.silver.tb_faturas',
    'tabela_pendentes': 'v-energy.silver.tb_faturas_pendentes_cadastro',
    'tabela_cadastro': 'workspace.default.silver_unidades_consumidoras',
    'tabela_leituras': 'venergy.silver.leituras',
    'schema_gold': 'v-energy.gold',
    'data_referencia': '',
    'diretorio_exportacao': '',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise ValueError('modo deve ser validar ou publicar')
date.fromisoformat(config['data_referencia'])


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise ValueError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
if config['schema_gold'].split('.')[1] in {'silver', 'bronze', 'default'}:
    raise ValueError('O destino deve ser um schema próprio da Gold')
if config['modo'] == 'publicar':
    caminho = Path(config['diretorio_exportacao'])
    if not str(caminho).startswith('/Volumes/') or '..' in caminho.parts:
        raise ValueError(
            'Publicação exige diretorio_exportacao em /Volumes/catalogo/schema/volume/...'
        )
batch_id = uuid4().hex
fontes = {}

## Leitura consistente da Silver

Os nomes físicos ainda diferem entre os notebooks Silver do projeto. Os widgets tornam essas referências explícitas. As duas saídas de faturas são reunidas para reconciliar tudo o que entra e o que é excluído da receita.

In [ ]:
def ler_snapshot(chave):
    tabela = qualificar(config[chave])
    detalhe = spark.sql(f'DESCRIBE DETAIL {tabela}').first()
    if detalhe['format'] != 'delta':
        raise ValueError(f'A fonte precisa ser Delta: {tabela}')
    historico = spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()
    fontes[chave] = {
        'tabela': config[chave],
        'id_delta': detalhe['id'],
        'versao_delta': historico['version'],
        'timestamp_commit': str(historico['timestamp']),
    }
    return spark.read.option('versionAsOf', historico['version']).table(tabela)


principal = ler_snapshot('tabela_faturas')
pendentes = ler_snapshot('tabela_pendentes')
cadastro = ler_snapshot('tabela_cadastro')
leituras = ler_snapshot('tabela_leituras')
if principal.filter(F.col('flag_uc_sem_cadastro')).limit(1).count():
    raise ValueError('A saída principal da Silver contém pendências cadastrais')
if pendentes.filter(~F.col('flag_uc_sem_cadastro')).limit(1).count():
    raise ValueError('A saída de pendências da Silver contém faturas com cadastro')
colunas_fonte = COLUNAS_FATURAS + [c for c in COLUNAS_AUDITORIA if c in principal.columns]
fonte_faturas = principal.select(*colunas_fonte).unionByName(pendentes.select(*colunas_fonte))
tabelas = construir_faturas(fonte_faturas, cadastro, leituras, config['data_referencia'])
validar_reconciliacao(fonte_faturas, tabelas)
if not tabelas['faturas'].limit(1).count():
    raise ValueError('Nenhuma fatura elegível: revise a integração com o cadastro Silver')

## Primeiro ano × últimos 12 meses

O intervalo deriva das competências presentes, e cada mês é exigido. Efeitos de preço e volume usam somente leituras utilizáveis e receita pareada. A variação da receita restante aparece separadamente. O preço efetivo inclui tributos, bandeira e mix: não é a tarifa homologada.

In [ ]:
mensal = tabelas['faturamento_mensal']
limites = mensal.agg(
    F.min('competencia').alias('inicio'),
    F.max('competencia').alias('fim'),
).first()
inicio, fim = limites['inicio'], limites['fim']


def deslocar_mes(data, meses):
    indice = data.year * 12 + data.month - 1 + meses
    return date(indice // 12, indice % 12 + 1, 1)


if deslocar_mes(inicio, 23) > fim:
    raise ValueError('A comparação anual exige ao menos 24 meses de histórico')
tabelas['variacao_receita'] = comparar_periodos(
    mensal,
    inicio.isoformat(),
    deslocar_mes(inicio, 11).isoformat(),
    deslocar_mes(fim, -11).isoformat(),
    fim.isoformat(),
)
_validacao_delta = tabelas['variacao_receita'].filter(
    F.col('variacao_receita')
    != F.col('efeito_volume') + F.col('efeito_preco_efetivo') + F.col('efeito_receita_sem_volume')
)
if _validacao_delta.limit(1).count():
    raise ValueError('Decomposição não reconcilia a variação de receita')
display(tabelas['faturamento_mensal'].orderBy('competencia', *DIMENSOES))
display(tabelas['variacao_receita'].orderBy('variacao_receita'))
display(
    tabelas['faturas_excluidas']
    .groupBy('motivo_exclusao')
    .agg(
        F.count('*').alias('faturas'),
        F.sum('valor_total').alias('valor_total'),
    )
)

## Publicação Delta e exportação CSV

A publicação só começa depois das validações. As tabelas Delta são substituídas sequencialmente; não existe transação única entre tabelas. O Job limita a concorrência a uma execução. A exportação usa um diretório exclusivo por lote e escreve o manifesto por último, após conferir as tabelas persistidas. O formato é próprio deste domínio; não representa o contrato provisório de 12 tabelas do portal.

In [ ]:
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema, tabela ou CSV foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise ValueError('Parâmetros mudaram. Execute novamente desde o início.')
    destinos = {nome: f'{schema_gold}.`tb_{nome}`' for nome in tabelas}
    for chave, fonte in fontes.items():
        atual = spark.sql(f'DESCRIBE DETAIL {qualificar(config[chave])}').first()
        if atual['id'] != fonte['id_delta']:
            raise ValueError(f'A identidade da fonte mudou: {chave}')
    if set(destinos.values()) & {qualificar(config[c]) for c in fontes}:
        raise ValueError('Uma tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas = {}
    versoes_gold = {}
    for nome, df in tabelas.items():
        destino = destinos[nome]
        df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(
            destino
        )
        versao = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
        persistido = spark.read.option('versionAsOf', versao).table(destino)
        if df.schema != persistido.schema:
            # Delta pode relaxar nulabilidade, mas nunca nomes/tipos/ordem.
            schema_esperado = [(c.name, c.dataType.simpleString()) for c in df.schema]
            schema_gravado = [(c.name, c.dataType.simpleString()) for c in persistido.schema]
            if schema_esperado != schema_gravado:
                raise ValueError(f'Schema persistido divergente: {destino}')
        if df.exceptAll(persistido).limit(1).count() or persistido.exceptAll(df).limit(1).count():
            raise ValueError(f'Conteúdo persistido divergente: {destino}')
        persistidas[nome] = persistido
        versoes_gold[nome] = {'tabela': destino, 'versao_delta': versao}
    validar_reconciliacao(fonte_faturas, persistidas)

    pasta_lote = exportar_csv(
        persistidas,
        str(Path(config['diretorio_exportacao']) / batch_id),
        {
            'batch_id': batch_id,
            'synthetic': False,
            'data_referencia': config['data_referencia'],
            'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
            'fontes': fontes,
            'versoes_gold': versoes_gold,
        },
    )
    print(f'Gold publicada e CSVs disponíveis em {pasta_lote}')